In [2]:
from pathlib import Path
import logging

import pandas as pd
from sqlalchemy import create_engine

from ingestion_db import ingest_db


Path("logs").mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    filename="logs/get_vendor_summary.log",
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s",
    filemode="a",
)


def create_vendor_summary(engine):
    """Merge purchase, sales, and vendor invoice tables into a vendor summary."""
    query = """
    WITH FreightSummary AS (
        SELECT VendorNumber, SUM(Freight) AS FreightCost
        FROM vendor_invoice
        GROUP BY VendorNumber
    ),
    PurchaseSummary AS (
        SELECT
            p.VendorNumber,
            p.VendorName,
            p.Brand,
            p.Description,
            p.PurchasePrice,
            pp.Price AS ActualPrice,
            pp.Volume,
            SUM(p.Quantity) AS TotalPurchaseQuantity,
            SUM(p.Dollars) AS TotalPurchaseDollars
        FROM purchase p
        JOIN purchase_prices pp ON p.Brand = pp.Brand
        WHERE p.PurchasePrice > 0
        GROUP BY
            p.VendorNumber,
            p.VendorName,
            p.Brand,
            p.Description,
            p.PurchasePrice,
            pp.Price,
            pp.Volume
    ),
    SalesSummary AS (
        SELECT
            VendorNo,
            Brand,
            SUM(SalesQuantity) AS TotalSalesQuantity,
            SUM(SalesDollars) AS TotalSalesDollars,
            SUM(SalesPrice) AS TotalSalesPrice,
            SUM(ExciseTax) AS TotalExciseTax
        FROM sales
        GROUP BY VendorNo, Brand
    )
    SELECT
        ps.VendorNumber,
        ps.VendorName,
        ps.Brand,
        ps.Description,
        ps.PurchasePrice,
        ps.ActualPrice,
        ps.Volume,
        ps.TotalPurchaseQuantity,
        ps.TotalPurchaseDollars,
        ss.TotalSalesQuantity,
        ss.TotalSalesDollars,
        ss.TotalSalesPrice,
        ss.TotalExciseTax,
        fs.FreightCost
    FROM PurchaseSummary ps
    LEFT JOIN SalesSummary ss
        ON ps.VendorNumber = ss.VendorNo
        AND ps.Brand = ss.Brand
    LEFT JOIN FreightSummary fs
        ON ps.VendorNumber = fs.VendorNumber
    ORDER BY ps.TotalPurchaseDollars DESC
    """
    return pd.read_sql_query(query, engine)


def clean_data(df):
    """Clean data and create derived metrics."""
    df["Volume"] = pd.to_numeric(df["Volume"], errors="coerce")
    df = df.fillna(0)

    df["VendorName"] = df["VendorName"].astype(str).str.strip()
    df["Description"] = df["Description"].astype(str).str.strip()

    df["GrossProfit"] = df["TotalSalesDollars"] - df["TotalPurchaseDollars"]

    sales = df["TotalSalesDollars"].where(df["TotalSalesDollars"] != 0)
    purchases = df["TotalPurchaseDollars"].where(
        df["TotalPurchaseDollars"] != 0
    )
    purchase_quantity = df["TotalPurchaseQuantity"].where(
        df["TotalPurchaseQuantity"] != 0
    )

    df["ProfitMargin"] = df["GrossProfit"].div(sales) * 100
    df["StockTurnover"] = df["TotalSalesQuantity"].div(purchase_quantity)
    df["SalesToPurchaseRatio"] = df["TotalSalesDollars"].div(purchases)

    return df


if __name__ == "__main__":
    engine = create_engine("sqlite:///inventory.db")

    try:
        logging.info("Creating Vendor Summary Table...")
        summary_df = create_vendor_summary(engine)
        logging.info(summary_df.head())

        logging.info("Cleaning Data...")
        clean_df = clean_data(summary_df)
        logging.info(clean_df.head())

        logging.info("Ingesting Data...")
        ingest_db(clean_df, "vendor_sales_summary", engine)

        logging.info("Completed Successfully")
    finally:
        engine.dispose()